In [2]:
import torch
from torch.utils.data import DataLoader, TensorDataset, random_split
from collections import OrderedDict
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import scanpy as sc
import numpy as np
import pandas as pd
from tqdm import tqdm
import csv
from scipy.sparse import issparse


In [ ]:
def preprocess_data(adata, immune_cell, n_genes, resolution):
    # Read the data
    if immune_cell == 'tcell':
        immune_cell = 'T'
    elif immune_cell == 'bcell':
        immune_cell = 'B'
    else:
        raise ValueError('Invalid immune cell type')

    # Ensure adata is not a view
    adata = adata.copy()


    # Filter the tumor cells
    tumor_cells = adata[adata.obs['cell_type'].astype(int) == 1].copy()

    # Debug: Check tumor cells
    print(f"Tumor cells shape after filtering: {tumor_cells.shape}")
    if tumor_cells.shape[0] == 0:
        print("Warning: No tumor cells found after filtering.")

    # Check if the expression matrix is sparse and convert to dense if necessary
    if issparse(tumor_cells.X):
        tumor_cells_X_dense = tumor_cells.X.toarray()
    else:
        tumor_cells_X_dense = tumor_cells.X

    # Calculate mean expression
    mean_expression = tumor_cells_X_dense.mean(axis=0)

    # Select top n genes
    print(f"Selecting top {n_genes} genes based on mean expression")
    top_n_genes = mean_expression.argsort()[-n_genes:][::-1]

    adata = adata[:, top_n_genes].copy()

 
    
    adata.obs[immune_cell] = adata.obs[immune_cell].astype(float)
    tumor_cells.obs[immune_cell] = tumor_cells.obs[immune_cell].astype(float)
    
    # Binarize the immune cell column based on the percentile value if resolution is not 'high'
    if resolution != 'high':
        if tumor_cells.obs[immune_cell].empty:
            print(f"Error: tumor_cells.obs[{immune_cell}] is empty.")
        else:
            percentile_value = np.percentile(tumor_cells.obs[immune_cell], 50)
            print(f"Percentile value: {percentile_value}")
            adata.obs[immune_cell] = np.where(adata.obs[immune_cell] > percentile_value, 1, 0)
            print(f"adata.obs[{immune_cell}] after binarization: {adata.obs[immune_cell].head()}")

    return adata


class BagsDataset(Dataset):
    def __init__(self, input_data, immune_cell, max_instances=None, radius=200, resolution='low',n_genes=500):
        self.immune_cell = immune_cell
        self.max_instances = max_instances
        self.radius = radius
        self.resolution = resolution
        self.n_genes = n_genes
        if isinstance(input_data, str):
            self.bags = self.create_bags_from_csv(input_data)
        elif isinstance(input_data, sc.AnnData):
            input_data = preprocess_data(input_data, immune_cell,n_genes,self.resolution)
            print(f"Preprocessed data: {input_data.X.shape}")
            self.bags = self.create_bags_from_adata(input_data)
        else:
            raise ValueError("input_data must be either a path to a CSV file or an AnnData object")

    def __len__(self):
        return len(self.bags)

    def __getitem__(self, idx):
        bag = self.bags[idx]
        distances = torch.tensor(bag['distances'], dtype=torch.float32)
        gene_expression = bag['gene_expression']
        if sp.issparse(gene_expression):
            gene_expression = torch.tensor(gene_expression.todense(), dtype=torch.float32)
        else:
            gene_expression = torch.tensor(gene_expression, dtype=torch.float32)
        label = torch.tensor(bag['label'], dtype=torch.float32)
        core_idx = bag['core_idx']
        gene_names = bag['gene_names']
        return distances, gene_expression, label, core_idx, gene_names

    def create_bags_from_csv(self, csv_file):
        data = pd.read_csv(csv_file)
        adata_radius_list = []
        for _, row in data.iterrows():
            adata_path = row['adata']
            resolution = row['resolution'] if 'resolution' in row and not pd.isna(row['resolution']) else self.resolution
            adata = sc.read_h5ad(adata_path)
            adata.obs_names_make_unique()
            adata
            adata = preprocess_data(adata, self.immune_cell, self.n_genes,resolution=resolution)
            radius = row['radius'] if 'radius' in row and not pd.isna(row['radius']) else self.radius
            adata_radius_list.append((adata, radius, resolution))
            print(f"Processing: adata={adata_path.split('/')[-1]}, radius={radius}, resolution={resolution}")
        return self.create_bags(adata_radius_list)

    def create_bags_from_adata(self, adata):
        adata_radius_list = [(adata, self.radius, self.resolution)]
        return self.create_bags(adata_radius_list)

    def create_bags(self, adata_radius_list):
        bags = {}
        bag_id = 0

        for adata, radius, resolution in adata_radius_list:
            spatial_coords_x = adata.obs['X'].astype(float)
            spatial_coords_y = adata.obs['Y'].astype(float)
            spatial_coords = np.array(list(zip(spatial_coords_x, spatial_coords_y)))
            gene_expression = adata.X
            if self.immune_cell == 'tcell':
                labels = adata.obs['T'].values
            elif self.immune_cell == 'bcell':
                labels = adata.obs['B'].values
            else:
                raise ValueError("immune_cell must be either 'tcell' or 'bcell'")
            adata.obs['cell_type'] = adata.obs['cell_type'].astype(int)
            cell_types = adata.obs['cell_type'].values
            barcodes = adata.obs.index.values
            gene_names = adata.var_names.tolist()
            
            for i in trange(len(spatial_coords), desc=f"Creating Bags with radius {radius}", ncols=100, bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}{postfix}]"):
                if cell_types[i] == 0:
                    continue

                dist_matrix_row = cdist([spatial_coords[i]], spatial_coords, metric='euclidean')[0]
                in_circle = np.where(dist_matrix_row <= radius)[0]
                in_circle = [idx for idx in in_circle if cell_types[idx] == 1]
                """print(cell_types[i])
                print(cell_types[in_circle])
                print(labels[i])"""

                if resolution == 'high':
                    in_circle = [idx for idx in in_circle if idx != i]
                

                if len(in_circle) == 0:
                    continue

                if self.max_instances is not None and len(in_circle) > self.max_instances:
                    continue

                gene_data = gene_expression[in_circle]
                distances = np.asmatrix(dist_matrix_row[in_circle].reshape(-1, 1), dtype=np.float32)

                bags[bag_id] = {
                    'distances': distances,
                    'gene_expression': gene_data,
                    'label': labels[i],
                    'core_idx': i,
                    'gene_names': gene_names
                }

                bag_id += 1

        total_bags = len(bags)
        avg_instances_per_bag = sum(bags[i]['gene_expression'].shape[0] for i in bags) / total_bags if total_bags > 0 else 0
        print(f"Total bags created: {total_bags}")
        print(f"Average instances per bag: {avg_instances_per_bag:.0f}")

        return bags

def custom_collate_fn(batch):
    distances, gene_expressions, labels, core_idxs, gene_names_list = zip(*batch)
    distances = [torch.tensor(np.array(d), dtype=torch.float32) for d in distances]
    gene_expressions_tensors = []
    for g in gene_expressions:
        if sp.issparse(g):
            gene_expressions_tensors.append(torch.tensor(g.todense(), dtype=torch.float32))
        else:
            gene_expressions_tensors.append(g.clone().detach().float())
    labels = torch.tensor(labels, dtype=torch.float32)
    core_idxs = torch.tensor(core_idxs, dtype=torch.long)
    gene_names = gene_names_list
    return distances, gene_expressions_tensors, labels, core_idxs, gene_names

In [289]:
adata = sc.read_h5ad('/project/DPDS/Wang_lab/shared/spatial_TCR/data/train_validate/Visium/HumanBreastCancerA2/T_cell.h5ad')
dataset = BagsDataset(adata, immune_cell='tcell', radius=150, n_genes=5000, resolution='low')
train_size = int(0.7 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
#dataset = load_datasets(data_dir='/project/DPDS/Wang_lab/s439765/spatial_tcr/HumanLungCancerFFPE', immune_cell='tcell',radius=200,)

Tumor cells shape after filtering: (2067, 36601)
Selecting top 5000 genes based on mean expression
Percentile value: 2.979839563369751
adata.obs[T] after binarization: AAACAACGAATAGTTC-1    1
AAACAAGTATCTCCCA-1    1
AAACAATCTACTAGCA-1    1
AAACACCAATAACTGC-1    1
AAACAGAGCGACTCCT-1    1
Name: T, dtype: int64
Preprocessed data: (3986, 5000)


Creating Bags with radius 150: 100%|█████████████████████████| 3986/3986 [00:00<00:00, 13280.92it/s]

Total bags created: 2067
Average instances per bag: 5


In [290]:
adata.obs

,n_genes,leiden,X,Y,T,cell_type
AAACAACGAATAGTTC-1,5393,5,0,1600,3.957418,0
AAACAAGTATCTCCCA-1,4029,4,5000,10200,4.755109,0
AAACAATCTACTAGCA-1,6959,17,300,4300,3.779165,1
AAACACCAATAACTGC-1,7068,2,5900,1900,3.211151,1
AAACAGAGCGACTCCT-1,4322,4,1400,9400,4.373457,0
...,...,...,...,...,...,...
TTGTTTCACATCCAGG-1,5768,14,5800,4200,2.211655,1
TTGTTTCATTAGTCTA-1,6353,2,6000,3000,3.227344,1
TTGTTTCCATACAACT-1,4247,11,4500,2700,4.166192,0
TTGTTTGTATTACACG-1,6991,2,7300,4100,2.513989,1


In [291]:
dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=custom_collate_fn)
val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=1, shuffle=True, collate_fn=custom_collate_fn)

distances, gene_expressions, labels, core_index, current_genes = next(iter(dataloader))


In [292]:

class Distance(nn.Module):
    def __init__(self):
        super(Distance, self).__init__()
        self.a = nn.Parameter(torch.tensor(-3.0),requires_grad=True)
        #self.sparsemax = Sparsemax(dim=0)
        self.softmax = nn.Softmax(dim=0)

    
    def forward(self, x):
        #print(x)
        a = self.a
        x = self.softmax(-torch.exp(a) * x)
        return x


In [293]:
model = Distance()
output = model(distances[0])
print(output)

tensor([[8.7224e-04],
        [8.7224e-04],
        [8.7224e-04],
        [9.9651e-01],
        [8.7224e-04]], grad_fn=<SoftmaxBackward0>)


In [294]:
class Gene_expression(nn.Module):
    def __init__(self):
        super(Gene_expression, self).__init__()
        self.b = nn.Parameter(torch.tensor(1.0),requires_grad=True)
        #self.sparsemax = Sparsemax(dim=-1) 
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):

        b = self.b
        x = self.softmax(torch.exp(b) * x)
        return x

In [295]:
model = Gene_expression()
input_tensor = gene_expressions[0]
output = model(input_tensor)
print(output)

tensor([[3.7677e-02, 1.4081e-01, 3.5556e-02,  ..., 1.1168e-06, 6.1854e-07,
         6.1854e-07],
        [4.8944e-02, 1.3028e-01, 6.5492e-02,  ..., 3.3071e-06, 3.3071e-06,
         5.4850e-07],
        [3.4569e-02, 1.0728e-01, 1.1891e-01,  ..., 8.2911e-07, 8.2911e-07,
         8.2911e-07],
        [6.1080e-02, 2.0332e-01, 2.9322e-02,  ..., 1.2005e-06, 1.2005e-06,
         5.7738e-07],
        [5.1964e-02, 1.5133e-01, 8.1432e-02,  ..., 4.3544e-07, 8.5527e-07,
         8.5527e-07]], grad_fn=<SoftmaxBackward0>)


In [296]:
file_path = './data/tumor_antigens_fake.csv'
df = pd.read_csv(file_path)

# Extract gene names and HLA binding matrix
all_genes = df['Gene'].tolist()  # Gene names
hla_binder = torch.tensor(df.drop('Gene', axis=1).values, dtype=torch.float32)

In [297]:
hla_binder

tensor([[80., 12., 62.,  ..., 59., 39., 21.],
        [77., 86., 26.,  ..., 91., 49.,  7.],
        [ 2., 65.,  5.,  ..., 45., 40., 30.],
        ...,
        [66., 84., 87.,  ..., 11., 72., 16.],
        [79., 44., 73.,  ..., 32., 20., 56.],
        [26., 86., 82.,  ..., 99., 75., 61.]])

In [298]:
class Immunogenicity(nn.Module):
    def __init__(self, all_genes, hla_binder):
        super(Immunogenicity, self).__init__()
        self.all_genes = all_genes
        self.hla_binder = hla_binder  # HLA binding matrix (genes x HLA)
        
        # Learnable vector of length corresponding to the number of HLA types
        self.hla = nn.Parameter(torch.full((self.hla_binder.shape[1],), -1.0), requires_grad=True)
        
        # Map genes to indices for quick lookup
        self.gene_to_index = {gene: idx for idx, gene in enumerate(self.all_genes)}

    def forward(self, current_genes):
        # Filter genes to include only those present in all_genes
        filtered_genes = [gene for gene in current_genes if gene in self.gene_to_index]
        indices = [self.gene_to_index[gene] for gene in filtered_genes]
        
        if len(indices) == 0:
            return torch.zeros(0), []  # Return empty if no matching genes
        
        # Extract the HLA binding counts for the filtered genes
        #print(indices)
        gene_binding_matrix = self.hla_binder[indices]  # Shape: (len(filtered_genes), num_HLA)
        #print(gene_binding_matrix)
        # Perform element-wise multiplication with the learnable HLA vector
        hla_interaction = gene_binding_matrix * torch.exp(self.hla)  # Shape: (len(filtered_genes), num_HLA)
        
        # Sum over the HLA dimension to get the immunogenicity for each gene
        ig = torch.sum(hla_interaction, dim=1)  # Shape: (len(filtered_genes),)
        
        return ig, filtered_genes


In [299]:
model = Immunogenicity(all_genes, hla_binder)
output, filtered_genes = model(list(current_genes[0]))
print(len(output))
print(filtered_genes)

52
['CPNE7', 'MMP11', 'HIST1H4H', 'CNTD2', 'STC2', 'HIST3H2A', 'TOP2A', 'ARHGAP8', 'TNNT1', 'SNHG9', 'CXCL9', 'MMP9', 'RRM2', 'KLHDC7B', 'IL20', 'ALDH3B2', 'COMP', 'SYCP2', 'GINS2', 'ASCL2', 'COL2A1', 'CST1', 'RTP4', 'BIRC5', 'RPL39L', 'KRT37', 'CDK1', 'UBE2T', 'HEXIM2', 'HIST1H2BC', 'PCP2', 'GLI4', 'NUSAP1', 'TPX2', 'PVT1', 'ANO9', 'HIST1H3H', 'N4BP3', 'HOXC9', 'GATC', 'ZWINT', 'E2F1', 'SNHG3', 'PLEKHG4', 'CTXN1', 'TLCD1', 'MXD3', 'MYBL2', 'FBN2', 'CHST1', 'AURKB', 'MAPK8IP2']


In [300]:
len(filtered_genes)

52

In [59]:

class MIL(nn.Module):
    def __init__(self, all_genes, hla_binder):
        super(MIL, self).__init__()
        self.distance = Distance()
        self.gene_expression = Gene_expression()
        self.immunogenicity = Immunogenicity(all_genes, hla_binder)
        self.alpha = nn.Parameter(torch.tensor(1.0), requires_grad=True)
        self.beta = nn.Parameter(torch.tensor(1.0), requires_grad=True)
    
    def forward(self, distances, gene_expressions, current_genes):
        bag_outputs = []
        for distance, gene_expression in zip(distances, gene_expressions):
            distance = self.distance(distance)
            gene_expression = self.gene_expression(gene_expression)
            immunogenicity, filtered_genes = self.immunogenicity(current_genes)
            alpha = self.alpha
            beta = self.beta
        
            if len(filtered_genes) == 0:
                continue  # Skip if no overlapping genes
        
        # Print debug information
            #print(f"gene_expression shape: {gene_expression.shape}")
            #print(f"current_genes length: {len(current_genes)}")
            #print(f"filtered_genes length: {len(filtered_genes)}")
        
        
            gene_to_index = {gene: idx for idx, gene in enumerate(current_genes)}
        
            gene_indices = [gene_to_index[gene] for gene in filtered_genes if gene in gene_to_index]
            gene_expression = gene_expression[:, gene_indices]
        
            #print(f"Filtered gene_expression shape: {gene_expression.shape}")
            #print(f"Immunogenicity shape: {immunogenicity.shape}")
        
            z = gene_expression @ immunogenicity
            #print(f"z shape: {z.shape}")
            z = z.unsqueeze(1)
            #print(f"z shape: {z.shape}")
            #print(f"distance shape: {distance}")
            bag_output = distance * z
            bag_output = torch.sum(bag_output, dim=0)
            bag_output = torch.exp(alpha) * bag_output + beta
            bag_output = torch.sigmoid(bag_output)
            #print(bag_output)
            bag_outputs.append(bag_output)
            #df = pd.DataFrame(bag_outputs)
            #df.to_csv('output.csv')
    
        
        return torch.stack(bag_outputs).squeeze(dim=1)
    

class EarlyStopping:
    def __init__(self, patience=5, delta=0.001):
        self.patience = patience
        self.delta = delta
        self.best_loss = np.inf
        self.counter = 0
        self.stopped_epoch = 0
        self.early_stop = False

    def __call__(self, val_loss, model, epoch):
        if val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.counter = 0
            torch.save(model.state_dict(), 'final_model.pth')
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                self.stopped_epoch = epoch

In [60]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')   

In [61]:
model = MIL(all_genes,hla_binder).to(device)
criterion = nn.BCELoss().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)
num_epochs = 100
patience = 5
early_stopping = EarlyStopping(patience=patience, delta=0.1)

In [62]:
for epoch in range(num_epochs):
    model.train() 
    running_loss = 0.0
    
    with tqdm(dataloader, unit="batch") as tepoch:
        for i, (distances, gene_expressions, label, core_idx, current_genes) in enumerate(tepoch):
            tepoch.set_description(f"Epoch {epoch+1}/{num_epochs}")

            optimizer.zero_grad()

            distances = torch.stack(distances).to(device)
            gene_expressions = torch.stack(gene_expressions).to(device)
            label = label.clone().detach().float().to(device)
            
            output = model(distances, gene_expressions, list(current_genes[0]))
            
            loss = criterion(output, label)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            tepoch.set_postfix(loss=loss.item())

    epoch_loss = running_loss / len(dataloader)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}')
    

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for val_distances, val_gene_expressions, val_label, _, val_current_genes in val_dataloader:
            val_distances = torch.stack(val_distances).to(device)
            val_gene_expressions = torch.stack(val_gene_expressions).to(device)
            val_label = val_label.clone().detach().float().to(device)
            val_output = model(val_distances, val_gene_expressions, list(val_current_genes[0]))
            val_loss += criterion(val_output, val_label).item()
    
    val_loss /= len(val_dataloader)
    print(f'Validation Loss: {val_loss:.4f}')

    """early_stopping(val_loss, model, epoch)
    if early_stopping.early_stop:
        print(f'Early stopping at epoch {epoch+1}')
        break"""
        

Epoch 1/100: 100%|██████████| 858/858 [00:02<00:00, 401.29batch/s, loss=0.27]  


Epoch [1/100], Loss: 1.0086
Validation Loss: 0.9691


Epoch 2/100: 100%|██████████| 858/858 [00:02<00:00, 421.09batch/s, loss=1.82]  


Epoch [2/100], Loss: 0.8803
Validation Loss: 0.8759


Epoch 3/100: 100%|██████████| 858/858 [00:02<00:00, 393.26batch/s, loss=1.07] 


...

Epoch [7/100], Loss: 0.7400
Validation Loss: 0.7503


Epoch 8/100: 100%|██████████| 858/858 [00:01<00:00, 460.30batch/s, loss=0.741]


Epoch [8/100], Loss: 0.7341
Validation Loss: 0.7440


Epoch 9/100:  44%|████▍     | 380/858 [00:00<00:01, 458.33batch/s, loss=0.665]


KeyboardInterrupt: 